# Detect face masks from the webcam

Run all cells to open the webcam window. Press **q** in that window to stop the stream and release the camera.

In [1]:
from pathlib import Path

from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import load_model
import numpy as np
import cv2

In [3]:
CAMERA_INDEX = 0
FRAME_WIDTH = 640
FACE_CONFIDENCE = 0.60
NMS_THRESHOLD = 0.30
MIN_FACE_SIZE = 24
MAX_FACE_HEIGHT_RATIO = 0.70
CLASS_CONFIDENCE = 0.65

In [4]:
def detect_and_predict_mask(
    frame, face_net, mask_net, confidence_threshold=0.60,
    nms_threshold=0.30, min_face_size=24, max_face_height_ratio=0.70
):
    h, w = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(
        frame, 1.0, (300, 300), (104.0, 177.0, 123.0)
    )
    face_net.setInput(blob)
    detections = face_net.forward()

    candidate_boxes = []
    candidate_scores = []
    for i in range(detections.shape[2]):
        confidence = float(detections[0, 0, i, 2])
        if confidence < confidence_threshold:
            continue

        box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
        start_x, start_y, end_x, end_y = box.astype("int")
        start_x, start_y = max(0, start_x), max(0, start_y)
        end_x, end_y = min(w - 1, end_x), min(h - 1, end_y)
        box_width = end_x - start_x
        box_height = end_y - start_y

        if box_width < min_face_size or box_height < min_face_size:
            continue
        aspect_ratio = box_width / box_height
        if not 0.55 <= aspect_ratio <= 1.35:
            continue
        if box_height > h * max_face_height_ratio:
            continue

        candidate_boxes.append([start_x, start_y, box_width, box_height])
        candidate_scores.append(confidence)

    if not candidate_boxes:
        return [], np.empty((0, 2), dtype="float32")

    kept_indices = cv2.dnn.NMSBoxes(
        candidate_boxes, candidate_scores, confidence_threshold, nms_threshold
    )
    faces = []
    locations = []
    for index in np.asarray(kept_indices).reshape(-1):
        start_x, start_y, box_width, box_height = candidate_boxes[int(index)]
        end_x = start_x + box_width
        end_y = start_y + box_height
        face = frame[start_y:end_y, start_x:end_x]
        if face.size == 0:
            continue

        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        face = cv2.resize(face, (224, 224))
        face = img_to_array(face)
        faces.append(preprocess_input(face))
        locations.append((start_x, start_y, end_x, end_y))

    if not faces:
        return [], np.empty((0, 2), dtype="float32")

    faces = np.array(faces, dtype="float32")
    predictions = mask_net.predict(faces, batch_size=32, verbose=0)
    return locations, predictions


In [6]:
project_dir = Path.cwd()
if not (project_dir / "face_detector").is_dir():
    project_dir = project_dir / "FaceMask-Detection"

face_detector_dir = project_dir / "face_detector"
prototxt_path = face_detector_dir / "deploy.prototxt"
weights_path = face_detector_dir / "res10_300x300_ssd_iter_140000.caffemodel"
model_path = project_dir / "mask_detector.h5"

for required_path in (prototxt_path, weights_path, model_path):
    if not required_path.is_file():
        raise FileNotFoundError(f"Required model file not found: {required_path.resolve()}")

print("[INFO] loading face detector model...")
face_net = cv2.dnn.readNet(str(prototxt_path), str(weights_path))

print("[INFO] loading face mask detector model...")
mask_net = load_model(str(model_path), compile=False)

camera = cv2.VideoCapture(CAMERA_INDEX)
if not camera.isOpened():
    raise RuntimeError(
        f"Could not open camera {CAMERA_INDEX}. Check camera permission and close other camera apps."
    )
print("[INFO] camera opened; press q in the video window to stop")

[INFO] loading face detector model...
[INFO] loading face mask detector model...


OpenCV: AVFoundation didn't find any attached Video Input Devices!
OpenCV: camera failed to properly initialize!


RuntimeError: Could not open camera 0. Check camera permission and close other camera apps.

In [ ]:
try:
    while True:
        success, frame = camera.read()
        if not success or frame is None:
            raise RuntimeError("The camera opened but did not return a frame.")

        source_height, source_width = frame.shape[:2]
        if source_width > FRAME_WIDTH:
            scale = FRAME_WIDTH / source_width
            frame = cv2.resize(
                frame, (FRAME_WIDTH, round(source_height * scale))
            )

        locations, predictions = detect_and_predict_mask(
            frame, face_net, mask_net, FACE_CONFIDENCE, NMS_THRESHOLD,
            MIN_FACE_SIZE, MAX_FACE_HEIGHT_RATIO
        )

        for box, prediction in zip(locations, predictions):
            start_x, start_y, end_x, end_y = box
            mask_score, no_mask_score = map(float, prediction)
            best_score = max(mask_score, no_mask_score)

            if best_score < CLASS_CONFIDENCE:
                label = f"Uncertain {best_score * 100:.0f}%"
                color = (0, 215, 255)
            elif mask_score > no_mask_score:
                label = f"Mask {mask_score * 100:.0f}%"
                color = (0, 255, 0)
            else:
                label = f"No Mask {no_mask_score * 100:.0f}%"
                color = (0, 0, 255)

            text_y = max(start_y - 10, 15)
            cv2.putText(
                frame, label, (start_x, text_y),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
            )
            cv2.rectangle(frame, (start_x, start_y), (end_x, end_y), color, 2)

        cv2.imshow("Face Mask Detection - press q to stop", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    camera.release()
    cv2.destroyAllWindows()
    print("[INFO] camera released")

In [ ]:
# Cleanup is handled by the try/finally block above.